# Tests for loading in PBF data not through unstable overpass

Problem: overpass-api.de has become so unstable that is is unusable. At most a few queries are possible per day. 

So, ideally, we would like to load in data via boundary polygons from a local pbf that encodes a big area like europe (downloaded from a provider like geofabrik).

## Switch overpass server approach

Tried these two as listed at https://wiki.openstreetmap.org/wiki/Overpass_API#Instances_with_global_data_coverage, both don't work:

ox.settings.overpass_url = "https://maps.mail.ru/osm/tools/overpass/api"  
ox.settings.overpass_url = "https://overpass.private.coffee/api"

## graph_from_pbf from osmnx v2.1.0.dev0@pbf approach

https://github.com/gboeing/osmnx/pull/1338#issuecomment-4788125791

In [ ]:
import osmnx as ox
ox.__version__

In [ ]:
tags = {"highway": ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "secondary"]}
G = ox.pbf.graph_from_pbf("../austria-260623.osm.pbf", tags)

In [ ]:
Gp = ox.projection.project_graph(G)
gdf_nodes, gdf_edges = ox.convert.graph_to_gdfs(Gp)
fig, ax = ox.plot.plot_graph(Gp, node_size=2)

Problem: Cannot read in only from a bounding polygon, but must read in the whole pbf

## pyrosm approach

In [ ]:
import pyrosm
pyrosm.__version__

In [ ]:
from pyrosm import OSM
from pyrosm import get_data

In [ ]:
%run -i "functions.py"
cityfilename = "test_cities.csv"

In [ ]:
with open('../cities/'+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {slugify(rows[0]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2]} for rows in reader}

In [ ]:
for cityid, city_info in cities.items():
    location = ox.geocoder.geocode_to_gdf(city_info["nominatim_query"])
    location = fill_holes(extract_relevant_polygon(cityid, shapely.geometry.shape(location['geometry'][0])))

    citydata = OSM('../austria-260623.osm.pbf', bounding_box=location)
    
    drive_net = citydata.get_network(network_type="driving")
    drive_net.plot()
    break

Problem: This works for the driving network, but it does not work for the protected bike infrastructure which needs custom filters